In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x8

In [2]:
# --- الخلية 2: الاستيراد والإصلاح الفوري (Import & Fix) ---
import sys
import os
import glob
import subprocess
import numpy as np
import pandas as pd
import cv2
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt

# --- إصلاح التثبيت: البحث عن SMP و YOLO وتثبيتهما بالقوة ---
print("🔧 جاري التحقق من المكتبات المفقودة وتثبيتها يدوياً...")

def force_install(package_name_part):
    # البحث عن ملف الـ whl الخاص بالمكتبة
    files = glob.glob(f"/kaggle/input/**/*{package_name_part}*.whl", recursive=True)
    if not files:
        print(f"⚠️ لم يتم العثور على ملف تثبيت لـ {package_name_part}!")
        return False
    
    # نأخذ أحدث ملف أو أقصر اسم (لضمان أنه ليس إضافة)
    target_whl = sorted(files, key=len)[0]
    
    try:
        # التثبيت المباشر
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            target_whl, 
            "--no-deps", 
            "--ignore-installed",
            "--quiet"
        ])
        print(f"✅ تم التثبيت اليدوي بنجاح: {os.path.basename(target_whl)}")
        return True
    except Exception as e:
        print(f"❌ فشل التثبيت اليدوي لـ {package_name_part}: {e}")
        return False

# محاولة استيراد SMP، وإذا فشل نثبته
try:
    import segmentation_models_pytorch as smp
except ImportError:
    print("⚠️ SMP غير موجود، جاري التثبيت...")
    # نبحث عن ملف يحتوي على 'segmentation' و 'models'
    if force_install("segmentation_models"):
        # يجب تحديث المسار ليشعر بايثون بالمكتبة الجديدة
        import site
        from importlib import reload
        reload(site)
        try:
            import segmentation_models_pytorch as smp
            print("🎉 تم استيراد SMP بنجاح بعد الإصلاح!")
        except ImportError:
            # محاولة أخيرة: إضافة مسار site-packages يدوياً (نادر الحدوث)
            sys.path.append(site.getsitepackages()[0])
            import segmentation_models_pytorch as smp

# محاولة استيراد Ultralytics (YOLO)
try:
    from ultralytics import YOLO
except ImportError:
    print("⚠️ YOLO غير موجود، جاري التثبيت...")
    if force_install("ultralytics"):
        from ultralytics import YOLO
        print("🎉 تم استيراد YOLO بنجاح بعد الإصلاح!")

# مكتبات إضافية مساعدة (tim, pretrainedmodels) يحتاجها SMP
try:
    import timm
except ImportError:
    force_install("timm")

# --- التحقق النهائي ---
print("-" * 30)
try:
    print(f"✅ SMP Version: {smp.__version__}")
    # اختبار سريع لـ YOLO
    model_test = YOLO("yolov8n.pt") # لن يحمل شيئاً، فقط للتأكد من الكلاس
    print(f"✅ YOLO Loaded Successfully")
except Exception as e:
    print(f"⚠️ تحذير: لا تزال هناك مشكلة في المكتبات، لكن سنحاول الاستمرار. الخطأ: {e}")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 الجهاز النشط: {DEVICE}")

🔧 جاري التحقق من المكتبات المفقودة وتثبيتها يدوياً...
⚠️ SMP غير موجود، جاري التثبيت...
✅ تم التثبيت اليدوي بنجاح: segmentation_models_pytorch-0.5.0-py3-none-any.whl


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

🎉 تم استيراد SMP بنجاح بعد الإصلاح!
------------------------------
✅ SMP Version: 0.5.0
⚠️ تحذير: لا تزال هناك مشكلة في المكتبات، لكن سنحاول الاستمرار. الخطأ: ❌  Download failure for https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt. Environment is not online.
🚀 الجهاز النشط: cuda


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- الخلية 22: المحرك البلاتيني (Fixed Formatting & ID Sanitization) ---
import gc
import torch
from ultralytics import YOLO
import pandas as pd
import numpy as np
import cv2
import os
import glob
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt
import segmentation_models_pytorch as smp
import csv

# --- 1. إعداد المسارات ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"⚡ حالة المسرع: {DEVICE}")

TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"
LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
SUBMISSION_FILE = "submission.csv"

# --- 2. تحميل النماذج ---
print("⚙️ جاري تحميل المحرك البلاتيني...")

YOLO_MODEL_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_MODEL_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

# تحميل YOLO
if os.path.exists(YOLO_MODEL_PATH):
    yolo_model = YOLO(YOLO_MODEL_PATH)
else:
    # Fallback
    fallback_path = "best.pt"
    if os.path.exists(fallback_path):
        yolo_model = YOLO(fallback_path)
    else:
        yolo_model = None
        print("⚠️ تحذير: YOLO غير موجود.")

# تحميل U-Net
unet_model = smp.Unet(encoder_name="resnet50", encoder_weights=None, in_channels=3, classes=1, decoder_attention_type="scse")

if os.path.exists(UNET_MODEL_PATH):
    unet_model.load_state_dict(torch.load(UNET_MODEL_PATH, map_location=DEVICE))
elif os.path.exists("best_model_phase10.pth"):
    unet_model.load_state_dict(torch.load("best_model_phase10.pth", map_location=DEVICE))
else:
    print(f"⚠️ خطأ: موديل U-Net غير موجود.")

unet_model.to(DEVICE)
unet_model.eval()

# --- 3. دوال المعالجة المساعدة (كما هي) ---
def apply_high_pass_filter(data, cutoff=0.5, fs=500, order=5):
    if len(data) < order * 3: return data
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, data)

def smart_einthoven_fix(leads):
    if 'I' in leads and 'II' in leads and 'III' in leads:
        l = min(len(leads['I']), len(leads['II']), len(leads['III']))
        I = leads['I'][:l]; II = leads['II'][:l]; III = leads['III'][:l]
        residual = (I + III) - II
        leads['II'][:l] += residual / 3.0
        leads['I'][:l]  -= residual / 3.0
        leads['III'][:l]-= residual / 3.0
    return leads

def fast_viterbi(prob_map):
    H, W = prob_map.shape
    cost = -np.log(prob_map + 1e-6)
    dp = np.zeros_like(cost); parent = np.zeros_like(cost, dtype=int)
    dp[:, 0] = cost[:, 0]; smooth_factor = 0.5 
    for t in range(1, W):
        prev_cost = dp[:, t-1]
        c_m1 = np.roll(prev_cost, 1); c_m1[0] = np.inf
        c_0 = prev_cost
        c_p1 = np.roll(prev_cost, -1); c_p1[-1] = np.inf
        stacked = np.stack([c_m1+smooth_factor, c_0, c_p1+smooth_factor])
        min_vals = np.min(stacked, axis=0)
        which_move = np.argmin(stacked, axis=0) 
        dp[:, t] = cost[:, t] + min_vals
        parent[:, t] = np.clip(np.arange(H) + (which_move - 1), 0, H-1)
    path = np.zeros(W, dtype=int)
    path[-1] = np.argmin(dp[:, -1])
    for t in range(W-2, -1, -1): path[t] = parent[path[t+1], t+1]
    return H - path 

def batch_predict_unet(crops, model):
    if not crops: return [], []
    target_h = 256
    processed_inputs = []; scales = []; shapes = []
    for img in crops:
        h, w = img.shape[:2]; scale = target_h / h; new_w = int(w * scale)
        img_rz = cv2.resize(img, (new_w, target_h))
        tens = torch.from_numpy(img_rz).permute(2, 0, 1).float() / 255.0
        processed_inputs.append(tens); scales.append(scale); shapes.append((target_h, new_w))
    max_w = int(np.ceil(max([s[1] for s in shapes]) / 32) * 32)
    batch_tensor = torch.zeros((len(processed_inputs), 3, target_h, max_w), dtype=torch.float32)
    for i, tens in enumerate(processed_inputs): batch_tensor[i, :, :tens.shape[1], :tens.shape[2]] = tens
    batch_tensor = batch_tensor.to(DEVICE)
    with torch.no_grad():
        preds_orig = torch.sigmoid(model(batch_tensor))
        preds_flip = torch.sigmoid(model(torch.flip(batch_tensor, [3])))
        final_preds = (preds_orig + torch.flip(preds_flip, [3])) / 2.0
    final_preds_np = final_preds.cpu().numpy()
    return [final_preds_np[i, 0, :shapes[i][0], :shapes[i][1]] for i in range(len(processed_inputs))], scales

def robust_multi_point_calibration(crop, default_val=25.0):
    if crop is None or crop.size == 0: return default_val
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)); gray = clahe.apply(gray)
    edges_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    edges_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sx = np.sum(np.abs(edges_x), axis=0); sy = np.sum(np.abs(edges_y), axis=1)
    peaks_x, _ = find_peaks(sx, distance=10, prominence=50)
    peaks_y, _ = find_peaks(sy, distance=10, prominence=50)
    grid_sizes = []
    if len(peaks_x) > 3: grid_sizes.extend(np.diff(peaks_x)[(np.diff(peaks_x)>10) & (np.diff(peaks_x)<100)])
    if len(peaks_y) > 3: grid_sizes.extend(np.diff(peaks_y)[(np.diff(peaks_y)>10) & (np.diff(peaks_y)<100)])
    if len(grid_sizes) >= 5:
        grid_sizes = np.array(grid_sizes)
        q1 = np.percentile(grid_sizes, 25); q3 = np.percentile(grid_sizes, 75); iqr = q3 - q1
        clean = grid_sizes[(grid_sizes >= q1 - 1.5*iqr) & (grid_sizes <= q3 + 1.5*iqr)]
        if len(clean) > 0: return (0.6 * np.median(clean)) + (0.4 * np.mean(clean))
    return default_val

def advanced_perspective_correction(image):
    if image is None: return None
    h, w = image.shape[:2]
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (w//2, h//2))
    edges = cv2.Canny(small, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=80, minLineLength=w//8, maxLineGap=10)
    if lines is None: return image
    angles = [np.degrees(np.arctan2(l[0][3]-l[0][1], l[0][2]-l[0][0])) for l in lines]
    valid_angles = [a for a in angles if abs(a) < 15]
    if len(valid_angles) < 5: return image
    median_angle = np.median(valid_angles)
    if abs(median_angle) > 0.5:
        M = cv2.getRotationMatrix2D((w//2, h//2), median_angle, 1.0)
        return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return image

def get_yolo_crops_with_fallback(image, model):
    if model:
        try:
            results = model.predict(image, verbose=False, conf=0.15, imgsz=640, device=DEVICE)
            found = {}
            for box in results[0].boxes.cpu().numpy():
                cid = int(box.cls[0]); conf = box.conf[0]
                if cid not in found or conf > found[cid]['conf']: found[cid] = box.xyxy[0]
            if len(found) >= 10: 
                crops = [None] * 12; h, w = image.shape[:2]
                for i in range(12):
                    if i in found:
                        x1,y1,x2,y2 = map(int, found[i]); pad=5
                        crops[i] = image[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)]
                    else: crops[i] = np.zeros((100, 100, 3), dtype=np.uint8)
                return crops
        except: pass
    h, w = image.shape[:2]; rh = h//4; cw = w//3
    return [image[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]

def preprocess_remove_grid_rgb(image_rgb):
    if image_rgb is None: return None
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, np.array([0, 50, 50]), np.array([10, 255, 255])) | \
           cv2.inRange(hsv, np.array([170, 50, 50]), np.array([180, 255, 255]))
    img_c = image_rgb.copy(); img_c[mask > 0] = (255, 255, 255)
    return img_c

# --- 5. التنفيذ الآمن (ID Formatting Fixes) ---
if os.path.exists(TEST_CSV_PATH): 
    test_df = pd.read_csv(TEST_CSV_PATH)
else: 
    test_df = pd.DataFrame({'id': ['001'], 'fs': [500], 'sig_len': [5000]})

paths = glob.glob(f"{IMAGE_DIR}/**/*.png") + glob.glob(f"{IMAGE_DIR}/**/*.jpg")
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in paths}

print(f"🚀 بدء الاستخراج (Format Fixing Mode)...")

with open(SUBMISSION_FILE, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['id', 'value'])

buffer_rows = []
BUFFER_SIZE = 50000

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing"):
    # [FIX 1] تنظيف الـ ID: إزالة الكسور العشرية إذا وجدت
    try:
        sid = str(int(row['id'])) 
    except:
        sid = str(row['id']) # في حال كان الـ ID نصياً يحتوي حروف

    # [FIX 2] التحقق من اسم عمود الطول (sig_len vs number_of_rows)
    target_fs = row.get('fs', 500.0)
    if pd.isna(target_fs): target_fs = 500.0
    
    if 'sig_len' in row:
        tlen = row['sig_len']
    elif 'number_of_rows' in row:
        tlen = row['number_of_rows']
    else:
        tlen = 10 * target_fs
        
    if pd.isna(tlen): tlen = int(round(10 * target_fs))
    else: tlen = int(tlen)

    path = path_map.get(sid)
    leads_vals = {}
    
    # المعالجة
    if path:
        try:
            img = cv2.imread(path); img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = advanced_perspective_correction(img)
            global_grid = robust_multi_point_calibration(img, 25.0)
            crops = get_yolo_crops_with_fallback(img, yolo_model)
            
            clean_crops = []; valid_indices = []
            for i, crop in enumerate(crops):
                if crop is not None and crop.size > 100:
                    clean_crops.append(preprocess_remove_grid_rgb(crop)); valid_indices.append(i)

            if clean_crops:
                prob_maps, scales = batch_predict_unet(clean_crops, unet_model)
                for idx_in_batch, real_idx in enumerate(valid_indices):
                    prob = prob_maps[idx_in_batch]; scale = scales[idx_in_batch]
                    crop_orig = clean_crops[idx_in_batch]; lname = LEAD_NAMES[real_idx]
                    local_grid = robust_multi_point_calibration(crop_orig, default_val=global_grid)
                    raw = fast_viterbi(prob)
                    g_sc = local_grid * scale; divider = g_sc * 10.0 if g_sc > 1.0 else 250.0
                    sig_mv = (raw - np.median(raw)) / divider
                    if len(sig_mv) > 15: sig_mv = savgol_filter(sig_mv, 11, 3)
                    if len(sig_mv) > 20: sig_mv = apply_high_pass_filter(sig_mv, cutoff=0.5, fs=target_fs)
                    if len(sig_mv) > 0: leads_vals[lname] = resample(sig_mv, tlen)
                    else: leads_vals[lname] = np.zeros(tlen)
            
            for l in LEAD_NAMES: 
                if l not in leads_vals: leads_vals[l] = np.zeros(tlen)
            leads_vals = smart_einthoven_fix(leads_vals)
            del img, crops, clean_crops, prob_maps
            
        except: leads_vals = {l: np.zeros(tlen) for l in LEAD_NAMES}
    else: leads_vals = {l: np.zeros(tlen) for l in LEAD_NAMES}

    # [FIX 3] ضمان التنسيق الصحيح للقيم (Floats) وعدم وجود NaN
    for l in LEAD_NAMES:
        s = leads_vals[l]
        if len(s) != tlen: s = np.zeros(tlen)
        s = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0) # تنظيف القيم
        
        # تنسيق القيم كـ Float بسيط لتجنب مشاكل الكتابة
        current_rows = [[f"{sid}_{i}_{l}", f"{val:.4f}"] for i, val in enumerate(s)]
        buffer_rows.extend(current_rows)

    if len(buffer_rows) >= BUFFER_SIZE:
        with open(SUBMISSION_FILE, 'a', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerows(buffer_rows)
        buffer_rows = [] 
        gc.collect() 

if buffer_rows:
    with open(SUBMISSION_FILE, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(buffer_rows)
    buffer_rows = []

print("✅ تم حفظ submission.csv بتنسيق سليم (Fixed IDs & Floats).")

⚡ حالة المسرع: cuda
⚙️ جاري تحميل المحرك البلاتيني...
🚀 بدء الاستخراج (Format Fixing Mode)...


Processing: 100%|██████████| 24/24 [00:31<00:00,  1.33s/it]

✅ تم حفظ submission.csv بتنسيق سليم (Fixed IDs & Floats).


In [5]:
# --- خلية 23: التحقق النهائي (Updated for CSV) ---

# 1. تحديث المسارات لتناسب الصيغة الجديدة CSV
MY_SUB_PATH = "submission.csv"
# عادةً ملف العينة يكون csv أيضاً
OFFICIAL_SAMPLE_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.csv" 

print("🔍 جاري التحقق من ملف CSV (Long Format)...")

try:
    # التأكد من وجود الملف أولاً
    if not os.path.exists(MY_SUB_PATH):
        raise FileNotFoundError("ملف submission.csv غير موجود! تأكد أن الخلية 22 انتهت بنجاح.")

    # القراءة بصيغة CSV
    my_sub = pd.read_csv(MY_SUB_PATH)
    print(f"✅ تم تحميل ملفك: {MY_SUB_PATH}")

    try:
        if os.path.exists(OFFICIAL_SAMPLE_PATH):
            sample_sub = pd.read_csv(OFFICIAL_SAMPLE_PATH)
            has_sample = True
        else:
            # محاولة قراءة parquet إذا لم يوجد csv كخطة بديلة للمقارنة فقط
            sample_sub = pd.read_parquet(OFFICIAL_SAMPLE_PATH.replace('.csv', '.parquet'))
            has_sample = True
    except:
        has_sample = False
        print("⚠️ لم نتمكن من قراءة ملف العينة الرسمي للمقارنة (لا يؤثر على الحل).")

    # التحقق من أسماء الأعمدة
    required_columns = {'id', 'value'}
    my_columns = set(my_sub.columns)
    
    if my_columns != required_columns:
        print(f"❌ خطأ قاتل: الأعمدة غير صحيحة!")
        print(f"الموجود لديك: {my_columns}")
        print(f"المطلوب: {required_columns}")
        raise ValueError("Column Mismatch")
    else:
        print("✅ أسماء الأعمدة صحيحة (id, value).")

    # التحقق من نوع البيانات (float)
    first_value = my_sub.iloc[0]['value']
    # في CSV أحياناً يتم قراءة الأرقام كنص، لذا نتأكد
    if not isinstance(first_value, (float, np.floating, int, np.integer)):
        print(f"⚠️ تحذير: نوع القيم قد لا يكون float مباشرة. النوع المكتشف: {type(first_value)}")
        print("جاري محاولة التحويل للتأكد...")
        try:
            float(first_value)
            print("✅ القيم قابلة للتحويل لأرقام بنجاح.")
        except:
            print("❌ القيم ليست أرقاماً!")
            raise ValueError("Value Type Error")
    else:
        print("✅ نوع البيانات (float/number) صحيح.")

    # التحقق من شكل الـ ID
    first_id = str(my_sub.iloc[0]['id'])
    print(f"🔍 عينة من الـ ID في ملفك: {first_id}")
    if "_" not in first_id:
        print("⚠️ تحذير: شكل الـ ID يبدو غريباً (لا يحتوي على underscore).")
    
    # التحقق من عدد الصفوف
    print(f"📊 إجمالي عدد الصفوف في ملفك: {len(my_sub)}")
    
    if has_sample:
        print(f"📊 إجمالي عدد الصفوف في Sample الرسمي: {len(sample_sub)}")
        if len(my_sub) == len(sample_sub):
            print("✅ تطابق تام في عدد الصفوف!")
        else:
            print("ℹ️ عدد الصفوف مختلف (وهذا طبيعي إذا كان الاختبار مخفياً Hidden Test Set).")
    
    if len(my_sub) % 12 == 0:
            print("✅ عدد الصفوف يقبل القسمة على 12 (منطقي لـ 12 leads).")
    else:
            print("⚠️ تحذير: عدد الصفوف لا يقبل القسمة على 12!")

    print("\n🎉 النتيجة النهائية: ملف submission.csv سليم 100% وجاهز للرفع.")
    print("توكل على الله واضغط Submit!")

except Exception as e:
    print(f"\n❌ حدث خطأ أثناء التحقق: {e}")

🔍 جاري التحقق من ملف CSV (Long Format)...
✅ تم تحميل ملفك: submission.csv
✅ أسماء الأعمدة صحيحة (id, value).
✅ نوع البيانات (float/number) صحيح.
🔍 عينة من الـ ID في ملفك: 1053922973_0_I
📊 إجمالي عدد الصفوف في ملفك: 900000
📊 إجمالي عدد الصفوف في Sample الرسمي: 75000
ℹ️ عدد الصفوف مختلف (وهذا طبيعي إذا كان الاختبار مخفياً Hidden Test Set).
✅ عدد الصفوف يقبل القسمة على 12 (منطقي لـ 12 leads).

🎉 النتيجة النهائية: ملف submission.csv سليم 100% وجاهز للرفع.
توكل على الله واضغط Submit!
